In [ ]:
# !pip install tensorflow==2.16.1
import json
import os
import sys
import asyncio
import argparse
from collections import defaultdict
import time

os.environ["JAX_PLATFORMS"] = "cpu"

import torch
import numpy as np
import jax.numpy as jnp
import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64


def decode_base64(encoded_str):
    decoded_bytes = base64.b64decode(encoded_str)
    decoded_str = decoded_bytes.decode('utf-8')
    return decoded_str

def encode_base64(decoded_str):
    # decoded_str = "opt_state.mu.params.token_embedder.embedding"
    encoded_string = base64.b64encode(decoded_str.encode('utf-8')).decode('utf-8')
    return encoded_string

model_size = '1.5B'

if model_size == '0.5B':
    # 0.5B
    vocab_size = 151936
    base_emb_dim = 896
    head_dim = 64
    base_num_query_heads = 14
    base_num_kv_heads = 2
    base_mlp_dim = 4864
    base_num_decoder_layers = 24
elif model_size == '3B':
    # 3B
    vocab_size = 151936
    base_emb_dim = 2048
    head_dim = 128
    base_num_query_heads = 16
    base_num_kv_heads = 2
    base_mlp_dim = 11008
    base_num_decoder_layers = 36
elif model_size == '1.5B':
    # 1.5B
    vocab_size = 151936
    base_emb_dim = 1536
    head_dim = 128
    base_num_query_heads = 12
    base_num_kv_heads = 2
    base_mlp_dim = 8960
    base_num_decoder_layers = 28

# 1.build model param keys
# 新保存一个jax版本的模型，从_sharding中读取模型的keys，之后也会基于这个模型对参数进行转换。如果已经有_sharding，可以直接读取
_sharding_path = 'gs://newproject-1-llm_base_models_europe-west4/qwen2.5/_sharding'
# sharding的模型是0.5B的，层数为24
sharding_file_model_layers= 24
last_layer_name = f'layers_{sharding_file_model_layers - 1}'
_sharding_path = epath.Path(_sharding_path)
with _sharding_path.open('r') as f:
    _sharding = json.load(f)

model_shardings = {}
for k, v in _sharding.items():
    base_k = decode_base64(k)
    base_k_split = base_k.split('.')
    if base_k_split.count('params') == 2:
        base_k_split = base_k_split[1: ]
    if 'opt_state' in base_k or 'step' in base_k: continue
    print(base_k_split)
    model_shardings[tuple(base_k_split)] = v
    
    if last_layer_name in base_k_split:
        assert base_k_split[2] == last_layer_name
        # 拓展额外的层
        for l in range(sharding_file_model_layers, base_num_decoder_layers, 1):
            base_k_split[2] = f'layers_{l}'
            print(base_k_split)
            model_shardings[tuple(base_k_split)] = v


2025-04-28 03:44:16.396609: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745811856.409467  197831 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745811856.413321  197831 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745811856.424494  197831 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745811856.424508  197831 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745811856.424511  197831 computation_placer.cc:177] computation placer alr

['params', 'token_embedder', 'embedding']
['params', 'decoder', 'decoder_norm', 'scale']
['params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_0', 'kernel']
['params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_1', 'kernel']
['params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wo', 'kernel']
['params', 'decoder', 'layers_4', 'sub_0', 'post_self_attention_layer_norm', 'scale']
['params', 'decoder', 'layers_4', 'sub_0', 'pre_self_attention_layer_norm', 'scale']
['params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'out', 'kernel']
['params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'bias']
['params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'kernel']
['params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'query', 'bias']
['params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'query', 'kernel']
['params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'value', 'bias']
['params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'value', 

In [ ]:
# 2.load torch model
# !pip install accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = f"Qwen/Qwen2.5-{model_size}B"  # 替换为你要下载的模型
cache_dir = f"/home/lishengping/{model_size}B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype="auto",
    device_map="auto"
)
# tokenizer = AutoTokenizer.from_pretrained(cache_dir)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:
# 3.convert model
qwen_jax2torch = {
    'params.token_embedder.embedding': 'model.embed_tokens.weight',  
    'params.decoder.decoder_norm.scale': 'model.norm.weight',
    'params.decoder.layers_0.sub_0.mlp.wi_1.kernel': 'model.layers.0.mlp.up_proj.weight', # .T
    'params.decoder.layers_0.sub_0.mlp.wi_0.kernel': 'model.layers.0.mlp.gate_proj.weight', # .T
    'params.decoder.layers_0.sub_0.mlp.wo.kernel': 'model.layers.0.mlp.down_proj.weight', # .T
    'params.decoder.layers_0.sub_0.post_self_attention_layer_norm.scale': 'model.layers.0.post_attention_layernorm.weight',
    'params.decoder.layers_0.sub_0.pre_self_attention_layer_norm.scale': 'model.layers.0.input_layernorm.weight',
    'params.decoder.layers_0.sub_0.self_attention.query.kernel': 'model.layers.0.self_attn.q_proj.weight', # .T.reshape(base_emb_dim, base_num_query_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.query.bias': 'model.layers.0.self_attn.q_proj.bias', # .reshape(base_num_query_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.key.kernel': 'model.layers.0.self_attn.k_proj.weight', # .T.reshape(base_emb_dim, base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.key.bias': 'model.layers.0.self_attn.k_proj.bias', # .reshape(base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.value.kernel': 'model.layers.0.self_attn.v_proj.weight', # .T.reshape(base_emb_dim, base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.value.bias': 'model.layers.0.self_attn.v_proj.bias', # # .reshape(base_num_kv_heads, head_dim)
    'params.decoder.layers_0.sub_0.self_attention.out.kernel': 'model.layers.0.self_attn.o_proj.weight', # .T.reshape(base_num_query_heads, head_dim, base_emb_dim)
    
}

all_layers_qwen_jax2torch = {}

for k, v in qwen_jax2torch.items():
    if 'layers_0' in k:
        for l in range(base_num_decoder_layers):
            newk = k.replace('layers_0', f'layers_{l}')
            newv = v.replace('layers.0', f'layers.{l}')
            all_layers_qwen_jax2torch[newk] = newv
    else:
        all_layers_qwen_jax2torch[k] = v

convert_params = {}
for k, v in model_shardings.items():
    jax_key = '.'.join(k)
    print(f'jax_key: {jax_key}')
    torch_key = all_layers_qwen_jax2torch[jax_key]
    torch_value = model.state_dict()[torch_key]
    if '.mlp.' in jax_key:
        torch_value = torch_value.T
    elif 'query.kernel' in jax_key:
        torch_value = torch_value.T.reshape(base_emb_dim, base_num_query_heads, head_dim)
    elif 'query.bias' in jax_key:
        torch_value = torch_value.reshape(base_num_query_heads, head_dim)   
    elif 'key.kernel' in jax_key or 'value.kernel' in jax_key:
        torch_value = torch_value.T.reshape(base_emb_dim, base_num_kv_heads, head_dim)
    elif 'key.bias' in jax_key or 'value.bias' in jax_key:
        torch_value = torch_value.reshape(base_num_kv_heads, head_dim)    
    elif 'out.kernel' in jax_key:
        torch_value = torch_value.T.reshape(base_num_query_heads, head_dim, base_emb_dim)
    # assert v.shape == torch_value.shape
    convert_params[k] = jnp.array(torch_value.to(torch.float32), dtype=jnp.bfloat16)
flatten_convert_params = unflatten_dict(convert_params)

jax_key: params.token_embedder.embedding
jax_key: params.decoder.decoder_norm.scale
jax_key: params.decoder.layers_4.sub_0.mlp.wi_0.kernel
jax_key: params.decoder.layers_4.sub_0.mlp.wi_1.kernel
jax_key: params.decoder.layers_4.sub_0.mlp.wo.kernel
jax_key: params.decoder.layers_4.sub_0.post_self_attention_layer_norm.scale
jax_key: params.decoder.layers_4.sub_0.pre_self_attention_layer_norm.scale
jax_key: params.decoder.layers_4.sub_0.self_attention.out.kernel
jax_key: params.decoder.layers_4.sub_0.self_attention.key.bias
jax_key: params.decoder.layers_4.sub_0.self_attention.key.kernel
jax_key: params.decoder.layers_4.sub_0.self_attention.query.bias
jax_key: params.decoder.layers_4.sub_0.self_attention.query.kernel
jax_key: params.decoder.layers_4.sub_0.self_attention.value.bias
jax_key: params.decoder.layers_4.sub_0.self_attention.value.kernel
jax_key: params.decoder.layers_5.sub_0.mlp.wi_0.kernel
jax_key: params.decoder.layers_5.sub_0.mlp.wi_1.kernel
jax_key: params.decoder.layers_5.su

In [ ]:
# 4.save model
checkpoint_dir = f'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_{model_size}_torch2jax_0515/checkpoints/0/items'
orbax_checkpointer = ocp.PyTreeCheckpointer()
orbax_checkpointer.save(checkpoint_dir, {"params": flatten_convert_params}, force=True)
print(f"Quantized params checkpoint saved at: {checkpoint_dir}")

I0428 03:45:15.992310  200447 google_auth_provider.cc:181] Running on GCE, using service account 626151558586-compute@developer.gserviceaccount.com


Quantized params checkpoint saved at: gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_3B_torch2jax_0428/checkpoints/0/items


In [6]:
# 5.验证
mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
devices = np.asarray(jax.devices()).reshape([1] * len(mesh_axes))
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = np.float32 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
for k, v in model_shardings.items():
    jointk = '.'.join(k)
    if 'embedding' in jointk:
        shape = (vocab_size, base_emb_dim)
    elif 'scale' in jointk:
        shape = (base_emb_dim, )
    elif 'mlp.wi_' in jointk:
        shape = (base_emb_dim, base_mlp_dim)
    elif 'mlp.wo.' in jointk:
        shape = (base_mlp_dim, base_emb_dim)
    elif 'query.kernel' in jointk:
        shape = (base_emb_dim, base_num_query_heads, head_dim)
    elif 'query.bias' in jointk:
        shape = (base_num_query_heads, head_dim, )
    elif 'key.kernel' in jointk or 'value.kernel' in jointk:
        shape = (base_emb_dim, base_num_kv_heads, head_dim)
    elif 'key.bias' in jointk  or 'value.bias' in jointk:
        shape = (base_num_kv_heads, head_dim, )
    elif 'out.kernel' in jointk:
        shape = (base_num_query_heads, head_dim, base_emb_dim)
    else:
        print(f'Unmatched params name: {jointk}')
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)           
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)

ckpt = epath.Path(checkpoint_dir)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)
# 如果restored只是一个带有模型名字的字典，没有具体的value矩阵，可以检查下abstract_unboxed_params是不是多了或者漏了params这个key
restored = ckptr.restore(
  ckpt, item={'params': abstract_unboxed_params}, transforms={}, restore_args={'params': restore_args}
)
# 验证load出来的模型是否正确
for k, v in flatten_dict(restored).items():
    print(k, v.shape)

('params', 'token_embedder', 'embedding') (151936, 2048)
('params', 'decoder', 'decoder_norm', 'scale') (2048,)
('params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_0', 'kernel') (2048, 11008)
('params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_1', 'kernel') (2048, 11008)
('params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wo', 'kernel') (11008, 2048)
('params', 'decoder', 'layers_4', 'sub_0', 'post_self_attention_layer_norm', 'scale') (2048,)
('params', 'decoder', 'layers_4', 'sub_0', 'pre_self_attention_layer_norm', 'scale') (2048,)
('params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'out', 'kernel') (16, 128, 2048)
('params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'bias') (2, 128)
('params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'kernel') (2048, 2, 128)
('params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'query', 'bias') (16, 128)
('params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'query', 'kernel') (2048, 16, 128)
(

('params', 'params', 'decoder', 'decoder_norm', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'mlp', 'wi_0', 'kernel') (2048, 11008)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'mlp', 'wi_1', 'kernel') (2048, 11008)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'mlp', 'wo', 'kernel') (11008, 2048)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'post_self_attention_layer_norm', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'pre_self_attention_layer_norm', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'self_attention', 'key', 'bias') (2, 128)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'self_attention', 'key', 'kernel') (2048, 2, 128)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'self_attention', 'out', 'kernel') (16, 128, 2048)
('params', 'params', 'decoder', 'layers_0', 'sub_0', 'self_attention', 'query', 'bias') (16, 128)
('params', 'params', 'decoder', 'layers_0', 'sub_0', '